In [1]:
from pyspark.sql import SparkSession

# 1. Initialize Spark Session with the MySQL JDBC dependency
spark = SparkSession.builder \
    .appName("Spark-MySQL-Connection") \
    .config("spark.jars.packages", "com.mysql:mysql-connector-j:9.3.0") \
    .getOrCreate()

# 2. Define MySQL connection properties
mysql_url = "jdbc:mysql://localhost:3306/source_db"  # Replace 'source_db' with your database name
connection_properties = {
    "user": "root",
    "password": "Taksh13",
    "driver": "com.mysql.cj.jdbc.Driver"
}

# 3. READ data from a MySQL table into a Spark DataFrame
table_name = "orders"

df = spark.read.jdbc(
    url=mysql_url, 
    table=table_name, 
    properties=connection_properties
)

# Show the data
df.show()

# 4. WRITE data back to a new or existing MySQL table (Optional)
# df.write.jdbc(
#     url=mysql_url, 
#     table="new_table_name", 
#     mode="append", # Options: overwrite, append, ignore, errorifexists
#     properties=connection_properties
# )

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/07/08 20:22:32 WARN Utils: Your hostname, TEJASs-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 10.166.201.233 instead (on interface en0)
26/07/08 20:22:32 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/opt/anaconda3/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/tejasjadhav/.ivy2.5.2/cache
The jars for the packages stored in: /Users/tejasjadhav/.ivy2.5.2/jars
com.mysql#mysql-connector-j added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-29aebb35-5006-4990-9991-7a7a90ec54f1;1.0
	confs: [default]
	found com.mysql#mysql-connector-j;9.3.0 in local-m2-cache
	found com.google.protobuf#protobuf-java;4.29.0 in local-m2-cache
:: resolution report :: resolve 54ms :: artifacts dl 1ms
	:: modules in use:
	com.

+-------+----------+----------+-----------+
|orderid|customerid| orderdate|totalamount|
+-------+----------+----------+-----------+
|  10248|       101|2024-09-15|      87.50|
|  10249|       102|2024-09-15|      12.99|
|  10250|       103|2024-09-16|     350.40|
|  10251|       104|2024-09-16|      21.00|
|  10252|       105|2024-09-17|      55.00|
|  10253|       106|2024-09-17|     125.60|
|  10254|       107|2024-09-18|       9.99|
|  10255|       108|2024-09-18|      45.20|
|  10256|       109|2024-09-19|      10.50|
|  10257|       110|2024-09-19|      18.00|
|  10258|       101|2024-09-20|     205.15|
|  10259|       102|2024-09-20|      33.75|
|  10260|       103|2024-09-21|      78.90|
|  10261|       104|2024-09-21|      15.00|
|  10262|       105|2024-09-22|      49.99|
|  10263|       106|2024-09-22|      22.50|
|  10264|       107|2024-09-23|      89.30|
|  10265|       108|2024-09-23|      14.99|
|  10266|       109|2024-09-24|      60.00|
|  10267|       110|2024-09-24| 

In [2]:
df.printSchema()

root
 |-- orderid: long (nullable = true)
 |-- customerid: string (nullable = true)
 |-- orderdate: date (nullable = true)
 |-- totalamount: string (nullable = true)



In [3]:
df.columns

['orderid', 'customerid', 'orderdate', 'totalamount']

In [4]:
df.count()

100

In [5]:
df.show(5) 

+-------+----------+----------+-----------+
|orderid|customerid| orderdate|totalamount|
+-------+----------+----------+-----------+
|  10248|       101|2024-09-15|      87.50|
|  10249|       102|2024-09-15|      12.99|
|  10250|       103|2024-09-16|     350.40|
|  10251|       104|2024-09-16|      21.00|
|  10252|       105|2024-09-17|      55.00|
+-------+----------+----------+-----------+
only showing top 5 rows


In [6]:
df.describe().show()

+-------+------------------+-----------------+------------------+
|summary|           orderid|       customerid|       totalamount|
+-------+------------------+-----------------+------------------+
|  count|               100|              100|               100|
|   mean|           10297.5|            105.5|59.189799999999984|
| stddev|29.011491975882016|2.886751345948131| 60.21910623311467|
|    min|             10248|              101|             10.50|
|    max|             10347|              110|             99.99|
+-------+------------------+-----------------+------------------+



In [7]:
df.createOrReplaceTempView("orders")
sql_df = spark.sql("SELECT orderid, customerid,orderdate,totalamount FROM orders WHERE totalamount > 30.00")

In [8]:
sql_df.count()

51

In [9]:
sql_df.write.csv('Top orders.csv', header=True, mode='overwrite')

In [10]:
sql_df.write.parquet('Top orders.parquet', mode='overwrite')

In [11]:
sql_df.write.json('Top orders.json', mode='overwrite')

to read query plans and understand how Spark executes your code.

In [12]:
sql_df.explain()

== Physical Plan ==
*(1) Filter (cast(totalamount#3 as double) > 30.0)
+- *(1) Scan JDBCRelation(orders) [numPartitions=1] [orderid#0L,customerid#1,orderdate#2,totalamount#3] PushedFilters: [*IsNotNull(totalamount)], ReadSchema: struct<orderid:bigint,customerid:string,orderdate:date,totalamount:string>




In [13]:
# Stop the Spark session
# spark.stop()

26/07/08 21:22:23 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 636452 ms exceeds timeout 120000 ms
26/07/08 21:22:23 WARN SparkContext: Killing executors is not supported by current scheduler.
26/07/08 21:22:24 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:342)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$